In [1]:
import sys
sys.path.append('../../Simulate/')
import importlib

In [2]:
import BSReadSim_queue
importlib.reload(BSReadSim_queue)
from BSReadSim_queue import BSReadSim

import ReadProcessor
importlib.reload(ReadProcessor)
from ReadProcessor import ReadProcessor

In [3]:
working_path = "/home/wbguo/iproject/BSReadSim/test/"
ref_fasta = working_path + 'data/ref/BSB_test.fa'
outdir = working_path + "outdir3/"
prefix = 'sim'
var_contig = "chr10"

In [4]:
self = BSReadSim(ref_fasta = ref_fasta, outdir=outdir, 
                 prefix=prefix, overwrite_db=True, num_reads=10**5,
                 gzip=False, shuffle=False)

Initiating experiment...
Initiating methylation profile...

[Initiating meth_db] for chr10...
Filling with beta distribution for chr10...
Processed 187408 sites from contig chr10

[Initiating meth_db] for chr11...
Filling with beta distribution for chr11...
Processed 187844 sites from contig chr11

[Initiating meth_db] for chr12...
Filling with beta distribution for chr12...
Processed 184249 sites from contig chr12

[Initiating meth_db] for chr13...
Filling with beta distribution for chr13...
Processed 142075 sites from contig chr13

[Initiating meth_db] for chr14...
Filling with beta distribution for chr14...
Processed 154158 sites from contig chr14

[Initiating meth_db] for chr15...
Filling with beta distribution for chr15...
Processed 2252 sites from contig chr15


../../Simulate/StreamReads.py:42: UserWarning: Fastq file exists, will overwrite... /home/wbguo/iproject/BSReadSim/test/outdir3//sim_1.fastq
  warnings.warn(f'Fastq file exists, will overwrite... {fastq_file}')
../../Simulate/StreamReads.py:42: UserWarning: Fastq file exists, will overwrite... /home/wbguo/iproject/BSReadSim/test/outdir3//sim_2.fastq
  warnings.warn(f'Fastq file exists, will overwrite... {fastq_file}')


In [5]:
pos_map, meth_arr, _  = self.meth_db.load_contig(var_contig)                              # [pos_map, meth_arr, status]
self.processor  = ReadProcessor(meth_arr= meth_arr,
                                pos_map = pos_map,
                                var_profile = self.var_profile,
                                experiment  = self.experiment)

# time the processing

In [6]:
read_pickle = working_path + "/pkl/read_pair_raw.pkl"
import pickle
from copy import deepcopy
# open a file, where you stored the pickled data
with open(read_pickle, 'rb') as FILE:
    _, read_pair0 = pickle.load(FILE)

In [7]:
read_pair_processed = self.processor.process_read_pair(deepcopy(read_pair0))

In [8]:
%timeit self.processor.process_read_pair(deepcopy(read_pair0))

2.41 ms ± 15.5 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [9]:
read_pickle2= working_path + "/pkl/read_pair_processsed.pkl"
with open(read_pickle2, 'wb') as FILE:
    pickle.dump(read_pair_processed, FILE)

# time each function

In [24]:
import random

In [25]:
read_pair = deepcopy(read_pair0)

In [26]:
%%timeit

read1_idx   = random.choice([0, 1]) 
pattern_idx = random.choice([0, 1]) if self.experiment.undirectional else read1_idx
strand_idx  = random.choice([0, 1]) if read_pair[0]['strand']<0 else read_pair[0]['strand']
read_pair[1-read1_idx]['read2'] = 1
read_pair[0]['conv'] = pattern_idx
read_pair[1]['conv'] = pattern_idx
read_pair[0]['strand'] = strand_idx
read_pair[1]['strand'] = strand_idx

1.61 µs ± 7.81 ns per loop (mean ± std. dev. of 7 runs, 1000000 loops each)


In [13]:
%timeit self.processor.mask_context(read_pair[0])

219 µs ± 20.1 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [14]:
%timeit self.processor.retrive_meth_db(read_pair[0])

59.2 µs ± 2.41 µs per loop (mean ± std. dev. of 7 runs, 10000 loops each)


In [15]:
%timeit self.processor.retrive_meth_db(read_pair[1])

64.5 µs ± 2.69 µs per loop (mean ± std. dev. of 7 runs, 10000 loops each)


In [16]:
%timeit self.processor.set_context_state(read_pair)

517 µs ± 1.36 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [17]:
%timeit self.processor.treat_bisulfite(read_pair[0])

278 µs ± 846 ns per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [18]:
%timeit self.processor.treat_bisulfite(read_pair[1])

280 µs ± 642 ns per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [19]:
%timeit self.processor.rev_complement(read_pair)

32.5 µs ± 90.1 ns per loop (mean ± std. dev. of 7 runs, 10000 loops each)


In [20]:
%timeit self.processor.add_qual_score(read_pair[0])

6.63 µs ± 23.4 ns per loop (mean ± std. dev. of 7 runs, 100000 loops each)


In [21]:
%timeit self.processor.add_qual_score(read_pair[1])

6.74 µs ± 13.5 ns per loop (mean ± std. dev. of 7 runs, 100000 loops each)


In [22]:
%timeit self.processor.add_seq_err(read_pair[0])

191 µs ± 1.13 µs per loop (mean ± std. dev. of 7 runs, 10000 loops each)


In [23]:
%timeit self.processor.add_seq_err(read_pair[1])

191 µs ± 1.4 µs per loop (mean ± std. dev. of 7 runs, 10000 loops each)


In [ ]:
import random
import numpy as np
import pickle

from copy import deepcopy
from scipy.stats import bernoulli
from typing import Dict

In [ ]:
from StreamReads import StreamReads
from StreamHTSIM import StreamHTSIM
from SetExperiment import SetExperiment
from SetMethylation import SetMethylation
from ReadProcessor import ReadProcessor
from LockedIterator import LockedIterator

from Bio import SeqIO

In [ ]:
working_path = "/home/wbguo/iproject/BSReadSim/test/"
ref_fasta = working_path + 'data/ref/chr21.fa'
outdir = working_path + "outdir/"

In [ ]:
ref_dict  = SeqIO.to_dict(SeqIO.parse(ref_fasta, "fasta"))

In [ ]:
expt_set = SetExperiment(ref_dict, outdir)

In [ ]:
# meth_set = SetMethylation(ref_dict=ref_dict, 
#                           outdir=outdir, overwrite_db=True)
meth_set = SetMethylation(ref_dict=ref_dict, 
                          meth_db_path = working_path + "outdir/pkl/",
                          outdir=outdir, overwrite_db=True)
# time difference from build denovo and load is not big?

In [ ]:
meth_db  = meth_set.meth_db
meth_db.create_share_arr(meth_set.arr_max_size)

In [ ]:
fastq_out= StreamReads(outdir=outdir, prefix='sim')

In [ ]:
expt_set.htsim_opts # currently, htsim will not faithfully get to N if only specify N

In [ ]:
sim_cmd = [expt_set.htsim_path, ref_fasta] + [str(item) for key_val in expt_set.htsim_opts.items() for item in key_val]

In [ ]:
read_gen = LockedIterator(StreamHTSIM(sim_cmd=sim_cmd, pair_end=expt_set.pair_end)) 

In [ ]:
var_contig, sim_data= next(read_gen)

In [ ]:
var_profile = meth_set.set_var_meth(var_contig, sim_data)

In [ ]:
meth_db.load_contig_share(var_contig)

In [ ]:
_, read_pair = next(read_gen)

# time process_read_pair

In [ ]:
read_pair

In [ ]:
import importlib
import ReadProcessor
importlib.reload(ReadProcessor)
from ReadProcessor import ReadProcessor

In [ ]:
self = ReadProcessor(meth_db.shared_pos_map, meth_db.shared_meth_arr, var_profile, expt_set)

In [ ]:
self.process_read_pair2(read_pair)

In [ ]:
_, read_pair = next(read_gen)

In [ ]:
%%timeit -n 1000
read_pair2 = deepcopy(read_pair) # otherwise it will change read_pair2 and raise error
self.process_read_pair2(read_pair2)

# run in a for loop to check if there is any error

In [ ]:
for _, read_pair in read_gen:
    self.process_read_pair2(read_pair)

# if there is an error

In [ ]:
read_pair = self.load_pickle('chr21_17746528_17746917_1')

In [ ]:
read_pair

In [ ]:
import random
read1_idx   = random.choice([0, 1]) 
pattern_idx = random.choice([0, 1]) if self.experiment.undirectional else read1_idx
strand_idx  = random.choice([0, 1]) if read_pair[0]['strand']<0 else read_pair[0]['strand']
read_pair[1-read1_idx]['read2'] = 1
read_pair[0]['conv'] = pattern_idx
read_pair[1]['conv'] = pattern_idx
read_pair[0]['strand'] = strand_idx
read_pair[1]['strand'] = strand_idx


In [ ]:
# mask the context
self.mask_context(read_pair[0])
self.mask_context(read_pair[1])

In [ ]:
# retrive methy profile
self.retrive_meth_db(read_pair[0])
self.retrive_meth_db(read_pair[1])

In [ ]:
read_pair

In [ ]:
read_rec = read_pair

In [ ]:
if self.experiment.pair_end:
    if read_rec[0]['inner_dist'] <= 0: # overlapped read pair: merge meth, get state, split
        overlap_idx = np.where(read_rec[0]['pos'] == read_rec[1]['pos'][0])
        if np.any(overlap_idx):
            for idx in overlap_idx:
                overlap_len = self.experiment.read_len - idx
                if read_rec[0]['pos'][idx:] == read_rec[1]['pos'][:overlap_len]:
                    break
        else:
            idx = self.experiment.read_len
        # it's okay to have idx=0, or np.where returns null
        comb_meth = np.concatenate((read_rec[0]['meth'][:idx],read_rec[1]['meth']), axis=0)
        comb_state= self.fetch_meth_state(comb_meth)
        read_rec[0]['ctx'][np.where(comb_state[:self.experiment.read_len])[0]] +=1
        read_rec[1]['ctx'][np.where(comb_state[-self.experiment.read_len:])[0]]+=1
    else:
        read1_state = self.fetch_meth_state(read_rec[0]['meth'])
        read_rec[0]['ctx'][np.where(read1_state)[0]] += 1
        read2_state = self.fetch_meth_state(read_rec[1]['meth'])
        read_rec[1]['ctx'][np.where(read2_state)[0]] += 1
else:
    read_state = self.fetch_meth_state(read_rec['meth'])
    read_rec['ctx'][np.where(read_state)[0]] += 1

In [ ]:
read_rec[1]['meth']

In [ ]:
# set methylation states
self.set_context_state(read_pair)

In [ ]:
import numpy as np
def retrive_meth_db(self, read_rec):
    '''retrive methylation levels from meth_db, append meth and pos to read_rec'''
    read_meth = np.zeros(self.experiment.read_len)
    read_pos  = read_rec['start'] + np.arange(self.experiment.read_len)
    site_flag = np.logical_not(read_rec['ctx'].mask)            # unmasked sites

    if np.any(site_flag):                                       # contain methylable bases
        arr_idx  = 2

        if read_rec['flag_pos']:                                # covers mutation position
            if self.experiment.asm_sim:
                arr_idx  = 4 if read_rec['flag_mut'] else 3

            if read_rec['n_indel']:                             # handle indel first (offset)
                read_pos += read_rec['ofs']
                indel_site= read_rec['cgr'] == 3
                read_meth[indel_site]= self.fetch_meth_val(read_pos[indel_site], arr_idx, 3)

            if read_rec['n_sub']:
                snp_site = site_flag & (read_rec['cgr'] == 1)   # snp methylable site
                read_meth[snp_site]  = self.fetch_meth_val(read_pos[snp_site],  arr_idx, 1)

            match_site = site_flag & (read_rec['cgr'] == 0)     # match methylable site
            read_meth[match_site] = self.fetch_meth_val(read_pos[match_site],arr_idx, 0)
        else:
            match_site = site_flag                              # SNP/INDEL free region
            read_meth[match_site] = self.fetch_meth_val(read_pos[match_site],arr_idx, 0)
    read_rec['meth']   = read_meth
    read_rec['pos']    = read_pos

    
def fetch_meth_val(self, pos_arr, arr_idx, var_type):
    '''fetch the methylation value from meth_db'''
    if var_type == 1:           # SNP
        meth_val = [self.var_profile[pos][0] for pos in pos_arr]
    elif var_type== 3:          # insertion sites will have the same coordinate
        insert_pos, pos_count = np.unique(pos_arr, return_counts=True) # uniq_pos, pos_count
        meth_val  = []
        for ix, pos in enumerate(insert_pos):
            try:
                insert_meth = list(self.var_profile[pos][0][:pos_count[ix]])
            except:
                insert_meth = [0]*pos_count[ix]
            meth_val += insert_meth
    else:                       # match
        meth_val  = list(self.meth_arr[self.pos_map[pos_arr], arr_idx])
    return meth_val

In [ ]:
retrive_meth_db(self, read_rec)

In [ ]:
read_meth = np.zeros(self.experiment.read_len)
read_pos  = read_rec['start'] + np.arange(self.experiment.read_len)
site_flag = np.logical_not(read_rec['ctx'].mask)            # unmasked sites

if np.any(site_flag):                                       # contain methylable bases
    arr_idx  = 2

    if read_rec['flag_pos']:                                # covers mutation position
        if self.experiment.asm_sim:
            arr_idx  = 4 if read_rec['flag_mut'] else 3

        if read_rec['n_indel']:                             # handle indel first (offset)
            read_pos += read_rec['ofs']
            indel_site= read_rec['cgr'] == 3
            read_meth[indel_site]= self.fetch_meth_val(read_pos[indel_site], arr_idx, 3)

        if read_rec['n_sub']:
            snp_site = site_flag & (read_rec['cgr'] == 1)   # snp methylable site
            read_meth[snp_site]  = self.fetch_meth_val(read_pos[snp_site],  arr_idx, 1)

        match_site = site_flag & (read_rec['cgr'] == 0)     # match methylable site
        read_meth[match_site] = self.fetch_meth_val(read_pos[match_site],arr_idx, 0)
    else:
        match_site = site_flag                              # SNP/INDEL free region
        read_meth[match_site] = self.fetch_meth_val(read_pos[match_site],arr_idx, 0)
read_rec['meth']   = read_meth
read_rec['pos']    = read_pos

In [ ]:
read_pos[match_site]

In [ ]:
arr_idx

In [ ]:
pos_arr = read_pos[match_site]
var_type = 0

In [ ]:
if var_type == 1:           # SNP
    meth_val = [self.var_profile[pos][0] for pos in pos_arr]
elif var_type== 3:          # insertion sites will have the same coordinate
    insert_pos, pos_count = np.unique(pos_arr, return_counts=True) # uniq_pos, pos_count
    meth_val  = []
    for ix, pos in enumerate(insert_pos):
        try:
            insert_meth = list(self.var_profile[pos][0][:pos_count[ix]])
        except:
            insert_meth = [0]*pos_count[ix]
        meth_val += insert_meth
else:                       # match
    meth_val  = list(self.meth_arr[self.pos_map[pos_arr], arr_idx])

In [ ]:
read_rec

In [ ]:
''.join(['ACGT'[i] for i in read_rec['seq']])

In [ ]:
pos_arr

In [ ]:
self.pos_map[pos_arr]

In [ ]:
fetch_meth_val(read_pos[match_site],arr_idx, 0)

In [ ]:
self.pos_map[pos_arr]

In [ ]:
self.meth_arr

In [ ]:
len(self.pos_map)

In [ ]:
from pympler import asizeof
read_pair = self.process_read_pair(sim_data)

asizeof.asizeof(read_pair)

In [ ]:
read_pair

In [ ]:
''.join(['ACGT'[i] for i in sim_data[0]['seq']])

In [ ]:
''.join(['ACGT'[i] for i in sim_data[1]['seq']])

In [ ]:
import numpy as np

x = np.array([i in (1,2) for i in sim_data[0]['seq']])
y = np.logical_not(sim_data[0]['ctx'].mask)

In [ ]:
x

In [ ]:
y

In [ ]:
np.logical_xor(x, y)

In [ ]:
import numpy as np
a = np.array([0,0,1,1,2])
u, c = np.unique(a, return_counts=True)
dup = u[c > 1]

In [ ]:
u

In [ ]:
c

In [ ]:
dup

In [ ]:
sim_data[0]['seq'].flags

In [ ]:
sim_data[0]['ctx'].flags

In [ ]:
sim_data[0]['cgr'].flags

In [ ]:
sim_data[0]['ofs'].flags

In [ ]:
sim_data[0]['ctx'].flags

# if changed reload

In [ ]:
import StreamReads
importlib.reload(StreamReads)
from StreamReads import StreamReads

In [ ]:
import pickle

with open('/home/wbguo/iproject/BSReadSim/test/data/test_read_pair.pickle', 'rb') as handle:
    read_pair = pickle.load(handle)

In [ ]:
fastq_out = StreamReads(outdir=outdir, prefix="sim", pair_end=True)

In [ ]:
fastq_out.output_reads(read_pair, 0, 0)

In [ ]:
fastq_out.close()

In [ ]:
htsim_arg= ['/home/wbguo/iproject/BSReadSim/HTSIM/htsim']

sim_dict = {'-N':1000,
            '-h':0, 
            '-T':0,
            '-1':100, 
            '-2':100,
            '-e':0.000,
            '-i':400,
            '-I':25,
            '-r':0.000, 
            '-R':0.15,
            '-X':0.15, 
            '-A':0.05,
            '-f':1,
           }

In [ ]:
sim_sub = htsim_arg + [str(item) for key_val in sim_dict.items() for item in key_val]

sim_cmd =  sim_sub + [ref_fasta]

In [ ]:
i = 0
for variant_contig, sim_data in StreamHTSIM(sim_cmd):    
    if variant_contig:
        print(variant_contig)
        continue
    else:
        i +=1
    if len(sim_data[0]['seq']) != len(sim_data[0]['ctx']) or len(sim_data[1]['seq']) != len(sim_data[1]['ctx']):
        print('sth wrong')
        break

In [ ]:
i # TODO

In [ ]:
sim_data